# On the `dual` function

The  JuMP `dual` function corresponds to a slightly different notion than what it is usually understood in linear programming, as illustrated below.

In [1]:
using JuMP
using LinearAlgebra
using Gurobi
using GLPK

Consider the following linear program.

In [2]:
m = Model()

@variable(m, x[1:3])

@constraint(m, c1, sum(x[i] for i = 1:2) >= 3)
@constraint(m, c2, -2x[1] + 2x[2] - 4x[3] <= 5)

@constraint(m, n2, x[2] >= 0)
@constraint(m, n3, x[3] <= 0)

@objective(m, Min, 4x[1]+2x[2]+x[3])

println(m)

Min 4 x[1] + 2 x[2] + x[3]
Subject to
 c1 : x[1] + x[2] >= 3.0
 n2 : x[2] >= 0.0
 c2 : -2 x[1] + 2 x[2] - 4 x[3] <= 5.0
 n3 : x[3] <= 0.0



In [3]:
set_optimizer(m, Gurobi.Optimizer)

optimize!(m)

Academic license - for non-commercial use only - expires 2021-10-29
Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads
Optimize a model with 4 rows, 3 columns and 7 nonzeros
Model fingerprint: 0x16138732
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [1e+00, 4e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 5e+00]
Presolve removed 4 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds
Optimal objective  6.500000000e+00

User-callback calls 32, time in user-callback 0.00 sec


In [4]:
value.(x)

3-element Vector{Float64}:
 0.25
 2.75
 0.0

We can use the function `dual` to directly obtain the values of the dual variables, but as explained on the page https://jump.dev/JuMP.jl/stable/manual/constraints/ the objective function is not taken into account as the function uses the concept of conic duality: https://jump.dev/MathOptInterface.jl/v0.9.1/apimanual/#Duals-1

In the case of a minimization program, we obtain the desired result.

In [5]:
[dual(c1) ; dual(c2)]

2-element Vector{Float64}:
  3.0
 -0.5

JuMP also proposes the `shadow_price` function that evaluates the change in the objective if we relax the constraint by one unit. The term *relaxation* is not precisely defined, and we can see that on a greater-than constraint the sign is the opposite of what we expect.

In [6]:
[shadow_price(c1) ; shadow_price(c2)]

2-element Vector{Float64}:
 -3.0
 -0.5

Let's explicitly form the dual program in order to validate our observations.

In [7]:
m = Model()

@variable(m, y[1:2])

@constraint(m, c1, y[1] - 2y[2] == 4)
@constraint(m, c2, y[1] + 2y[2] <= 2)
@constraint(m, c3, -4y[2] >= 1)

@constraint(m, n1, y[1] >= 0)
@constraint(m, n2, y[2] <= 0)

@objective(m, Max, 3y[1]+5y[2])

println(m)

Max 3 y[1] + 5 y[2]
Subject to
 c1 : y[1] - 2 y[2] == 4.0
 c3 : -4 y[2] >= 1.0
 n1 : y[1] >= 0.0
 c2 : y[1] + 2 y[2] <= 2.0
 n2 : y[2] <= 0.0



In [8]:
set_optimizer(m, Gurobi.Optimizer)

optimize!(m)

Academic license - for non-commercial use only - expires 2021-10-29
Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads
Optimize a model with 5 rows, 2 columns and 7 nonzeros
Model fingerprint: 0x7db6f978
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 4e+00]
Presolve removed 5 rows and 2 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds
Optimal objective  6.500000000e+00

User-callback calls 30, time in user-callback 0.00 sec


In [9]:
value.(y)

2-element Vector{Float64}:
  3.0
 -0.5

The problem is now a maximization one, and when we call the `dual` function, we get the opposite of the primal optimal solution.

In [10]:
[ dual(c1) ; dual(c2) ; dual(c3) ]

3-element Vector{Float64}:
 -0.25
 -2.75
 -0.0

Here, the shadow prices correspond to what we expect, but the negative sign on the last variable reflects that the corresponding inequality constraint is of the type greater than.

In [11]:
[ shadow_price(c1) ; shadow_price(c2) ; shadow_price(c3) ]

3-element Vector{Float64}:
  0.25
  2.75
 -0.0

We can also observe this behavior from the example given at https://jump.dev/JuMP.jl/stable/manual/constraints/

In [12]:
model = Model(Gurobi.Optimizer)
@variable(model, x)
@constraint(model, con, x <= 1)
@objective(model, Min, -2x)
optimize!(model)
dual(con)

Academic license - for non-commercial use only - expires 2021-10-29
Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads
Optimize a model with 1 rows, 1 columns and 1 nonzeros
Model fingerprint: 0x57acb05e
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 1 rows and 1 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds
Optimal objective -2.000000000e+00

User-callback calls 24, time in user-callback 0.00 sec


-2.0

In [13]:
shadow_price(con)

-2.0

In [14]:
 @objective(model, Max, 2x)

2 x

In [15]:
optimize!(model)

Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads
Optimize a model with 1 rows, 1 columns and 1 nonzeros
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds
Optimal objective  2.000000000e+00

User-callback calls 40, time in user-callback 0.00 sec


In [16]:
dual(con)

-2.0

In [17]:
shadow_price(con)

2.0

In [18]:
model = Model(Gurobi.Optimizer)
@variable(model, x)
@constraint(model, con, -x >= 1)
@objective(model, Min, -2x)
optimize!(model)

Academic license - for non-commercial use only - expires 2021-10-29
Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads
Optimize a model with 1 rows, 1 columns and 1 nonzeros
Model fingerprint: 0xdb4ec760
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 1 rows and 1 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds
Optimal objective  2.000000000e+00

User-callback calls 24, time in user-callback 0.00 sec


In [19]:
dual(con)

2.0

In [20]:
shadow_price(con)

-2.0

In [21]:
@objective(model, Max, 2x)
optimize!(model)

Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads
Optimize a model with 1 rows, 1 columns and 1 nonzeros
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds
Optimal objective -2.000000000e+00

User-callback calls 40, time in user-callback 0.00 sec


In [22]:
dual(con)

2.0

In [23]:
shadow_price(con)

2.0